# Movie Sentiments Dashboard

In [84]:
# ── Imports ───────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['dash', 'dash-bootstrap-components', 'plotly', 'pandas']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, ctx
import dash_bootstrap_components as dbc

print("Packages installed")

Packages installed


In [85]:
# ── Data ──────────────────────────────────────────────────────────

movies_df = pd.read_csv('tmdb_movie_enriched_scl_aggregate.csv')

movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')
movies_df['year'] = movies_df['release_date'].dt.year
movies_df['revenue'] = pd.to_numeric(movies_df['revenue'], errors='coerce')
movies_df['sentiment_mean'] = pd.to_numeric(movies_df['sentiment_mean'], errors='coerce')

# Keep only movies with valid year, positive revenue, sufficient votes, and known sentiment
movies_df = movies_df[
    movies_df['year'].between(1925, 2025) &
    (movies_df['revenue'] > 0) &
    (movies_df['vote_count'] >= 10) &
    movies_df['dominant_polarity'].notna()
].copy()

movies_df['sentiment_category'] = movies_df['sentiment_mean'].apply(
    lambda x: 'positive' if x >= 0 else 'negative'
)

# Narrative keywords only (excluding locations like New York etc)
keywords_df = pd.read_csv('tmdb_movie_keyword_pairs_scl_enriched.csv')
keywords_df = keywords_df[
    keywords_df['is_narrative'] &
    (keywords_df['keyword_type'] != 'location')
].copy()

print(f'{len(movies_df):,} movies, {len(keywords_df):,} narrative keyword pairs')
movies_df['sentiment_category'].value_counts()

11,521 movies, 728,027 narrative keyword pairs


sentiment_category
positive    5899
negative    5622
Name: count, dtype: int64

In [86]:
# ── Theme & Colors ────────────────────────────────────────────────

COLORS = {
    'positive': 'rgb(255, 122, 48)',
    'negative': 'rgb(61, 116, 182)',
    'neutral': '#D6D4CC',
    'bg': '#DDDDDD',
    'bg-light': '#f3f6f4',
    'text': '#36454F',
    'grid': 'rgba(0,0,0,0.08)',
}

layout_base = dict(
    plot_bgcolor=COLORS['bg-light'],
    paper_bgcolor=COLORS['bg'],
    font=dict(family='Inter, sans-serif', size=13, color=COLORS['text']),
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=80, r=40, t=100, b=60),
)

## Charts

In [87]:
def make_area_chart(df, title='Positivity Pays: Feel-good films lead at the box office'):
    """
    Creates a stacked area chart showing how positive vs negative films split the revenue over time.
    
    Takes yearly revenue totals for each sentiment, smooths them out with a 5-year rolling average,
    then converts to percentage of the total revenue.
    """
    sentiments = ['positive', 'negative']

    # Sum revenue by year and sentiment
    data = (df.groupby(['year', 'sentiment_category'])['revenue']
              .sum()
              .unstack(fill_value=0)
              .reindex(columns=sentiments, fill_value=0))
    
    # smooth out spikes, apply rolling(5)
    smoothed = data.rolling(5, min_periods=1, center=True).mean()

    plot_df = (smoothed.reset_index()
               .melt(id_vars='year', var_name='sentiment_category', value_name='revenue'))

    fig = px.area(plot_df, x='year', y='revenue',
                  color='sentiment_category',
                  color_discrete_map=COLORS,
                  pattern_shape='sentiment_category',
                  pattern_shape_map={'negative': '/', 'positive': '+'},
                  groupnorm='percent', # normalize each year to 100%
                  category_orders={'sentiment_category': ['negative', 'positive']})
    fig.update_traces(opacity=0.7, fillpattern_solidity=0.08)
    fig.update_layout(title=dict(text=title, x=0.5, font=dict(size=20, color=COLORS['text'])),
                      **layout_base, height=600)
    fig.update_xaxes(title='Year', showgrid=True, gridcolor=COLORS['grid'], zeroline=False)
    fig.update_yaxes(title='% of Total Revenue', range=[0, 100], ticksuffix='%')
    
     # not using legend; label regions directly on the chart
    fig.add_annotation(text='Positive', xref='paper', yref='paper',
                        x=0.95, y=0.85, showarrow=False,
                        font=dict(size=14, color=COLORS['text']))
    fig.add_annotation(text='Negative', xref='paper', yref='paper',
                        x=0.95, y=0.15, showarrow=False,
                        font=dict(size=14, color=COLORS['text']))
    return fig

In [ ]:
NUM_GENRES = 6

def make_genre_chart(df, year_range, selected_genre=None):
    """Diverging horizontal bar chart comparing positive/negative sentiment breakdown 
    for the top 6 genres. 
    
    update_charts: Clicking a genre cross-filters the keyword chart and treemap.
    """
    exploded = df.assign(genre=df['genres'].str.split(', ')).explode('genre')
    top_genres = exploded['genre'].value_counts().head(NUM_GENRES).index.tolist()
    exploded = exploded[exploded['genre'].isin(top_genres)]

    counts = exploded.groupby(['genre', 'sentiment_category']).size().unstack(fill_value=0)
    counts = counts.reindex(columns=['positive', 'negative'], fill_value=0)
    totals = counts.sum(axis=1)
    pcts = counts.div(totals, axis=0) * 100

    pcts = pcts.loc[top_genres]
    counts = counts.loc[top_genres]

    pcts['negative'] = pcts['negative'] * -1

    # customdata
    # https://plotly.com/python/hover-text-and-formatting/#adding-other-data-to-the-hover-with-customdata-and-a-hover-template

    # Dim unselected genres when a genre is active
    neg_opacities = [0.9 if (selected_genre is None or g == selected_genre) else 0.25 for g in top_genres]
    pos_opacities = [0.9 if (selected_genre is None or g == selected_genre) else 0.25 for g in top_genres]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=pcts.index, x=pcts['negative'],
        name='Negative', orientation='h',
        marker=dict(color=COLORS['negative'], opacity=neg_opacities),
        customdata=top_genres,
        text=[f"{c} ({p:.0f}%)" for c, p in zip(counts['negative'], counts['negative'] / totals.loc[top_genres] * 100)],
        textposition='inside',
        hoverinfo='none',
        legendrank=1,
    ))
    fig.add_trace(go.Bar(
        y=pcts.index, x=pcts['positive'],
        name='Positive', orientation='h',
        marker=dict(color=COLORS['positive'], opacity=pos_opacities),
        customdata=top_genres,
        text=[f"{c} ({p:.0f}%)" for c, p in zip(counts['positive'], counts['positive'] / totals.loc[top_genres] * 100)],
        textposition='inside',
        hoverinfo='none',
        legendrank=2,
    ))

    title = f"Genre Tug-of-War from {year_range[0]}-{year_range[1]}"
    if selected_genre:
        title += f"  ·  {selected_genre}"
    fig.update_layout(**layout_base)
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=18, color=COLORS['text'])),
        barmode='relative',
        hovermode='closest',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        yaxis=dict(autorange='reversed', automargin=True),
        margin=dict(l=0, r=0, t=60, b=30),
        height=500,
    )
    fig.update_xaxes(
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        tickmode='array',
        tickvals=[-100, -50, -25, 0, 25, 50, 100],
        ticktext=['100%', '50%', '25%', '0', '25%', '50%', '100%'],
        showgrid=False,
    )
    return fig

In [ ]:
NUM_KEYWORDS = 10

def make_keyword_chart(df, year_range, selected_movie_id=None):
    """Horizontal bar chart of the most frequent narrative keywords.
    shows only that movie's keywords.
    
    update_charts: clicking a keyword updates treemap
    """
    movie_ids = df['tmdb_movie_id']
    kw = keywords_df[keywords_df['tmdb_movie_id'].isin(movie_ids)]
    kw = kw[kw['scl_polarity'].isin(['positive', 'negative'])]

    # Treemap click: scopes to a single movie
    if selected_movie_id is not None:
        kw = kw[kw['tmdb_movie_id'] == selected_movie_id]

    top_kw = kw['name'].value_counts().head(NUM_KEYWORDS)
    top_names = top_kw.index.tolist()
    top_counts = top_kw.values

    if len(top_names) == 0:
        return go.Figure().update_layout(**layout_base, title=dict(text='No keywords found', x=0.5))

    polarity_map = kw.drop_duplicates('name').set_index('name')['scl_polarity']
    bar_colors = [COLORS.get(polarity_map.get(n, 'neutral'), COLORS['neutral']) for n in top_names]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=top_names,
        x=top_counts,
        orientation='h',
        marker=dict(color=bar_colors, opacity=0.9),
        text=top_counts,
        textposition='outside',
        hoverinfo='none',
    ))

    if selected_movie_id is not None:
        movie_title = df.loc[df['tmdb_movie_id'] == selected_movie_id, 'title'].iloc[0] if (df['tmdb_movie_id'] == selected_movie_id).any() else '?'
        title = f"Keywords for {movie_title}"
    else:
        title = f"Top Keywords from {year_range[0]}-{year_range[1]}"
    fig.update_layout(**layout_base)
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=18, color=COLORS['text'])),
        hovermode='closest',
        yaxis=dict(autorange='reversed', automargin=True),
        xaxis=dict(title='Movie Count', showgrid=True, gridcolor=COLORS['grid']),
        margin=dict(l=0, r=30, t=60, b=30),
        height=500,
    )
    return fig

In [ ]:
def make_treemap(df, year_range, selected_keyword=None):
    """
    Creates a treemap of highest gross movies broken down by sentiment.
    Box size = revenue. Color = sentiment score.
    
    update_charts: clicking a treemap movie updates keyword chart

    Based on Plotly Express treemap docs: https://plotly.com/python/treemaps/
    """
    sub = df.copy()

    # Filter to movies that contain the selected keyword
    if selected_keyword is not None:
        matching_ids = keywords_df.loc[keywords_df['name'] == selected_keyword, 'tmdb_movie_id'].unique()
        sub = sub[sub['tmdb_movie_id'].isin(matching_ids)]

    label = f'{year_range[0]}-{year_range[1]}'
    if sub.empty:
        return go.Figure().update_layout(**layout_base, title=dict(text='No movies found', x=0.5))

    top = pd.concat([
        sub[sub['sentiment_category'] == cat].nlargest(10, 'revenue')
        for cat in ['positive', 'negative'] if (sub['sentiment_category'] == cat).any()
    ]).copy()
    top['sentiment_label'] = top['sentiment_category'].str.capitalize()
    top['display_title'] = top['title'].fillna('Untitled')

    fig = px.treemap(
        top,
        path=[px.Constant(label), 'sentiment_label', 'display_title'],
        values='revenue',
        color='sentiment_mean',
        color_continuous_scale=[[0, COLORS['negative']], [0.5, '#ffffff'], [1, COLORS['positive']]],
        color_continuous_midpoint=0,
    )

    fig.update_traces(
        hovertemplate='<b>%{label}</b><br>Revenue: $%{value:,.0f}<br>Sentiment Score: %{color:.2f}<extra></extra>',
    )

    title = 'Top Movies by Revenue'
    if selected_keyword:
        title += f'  ·  "{selected_keyword}"'
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=18, color=COLORS['text'])),
        margin=dict(l=10, r=10, t=50, b=10),
        height=450,
        coloraxis_showscale=False,
    )
    return fig

In [91]:
# look up movie keyword memento
fg = movies_df[movies_df['title'] == 'Gone with the Wind'].iloc[0]
print(f"Title: {fg['title']}")
print(f"Sentiment Mean: {fg['sentiment_mean']:.4f}")
print(f"Sentiment Category: {fg['sentiment_category']}")
print(f"Revenue: ${fg['revenue']:,.0f}")
print()

fg_kw = keywords_df[keywords_df['tmdb_movie_id'] == fg['tmdb_movie_id']]
print(f"Keywords ({len(fg_kw)}):")
fg_kw[['name', 'keyword_type', 'scl_polarity', 'scl_valence']].sort_values('scl_valence')

Title: Gone with the Wind
Sentiment Mean: -0.2219
Sentiment Category: negative
Revenue: $402,352,579

Keywords (16):


,name,keyword_type,scl_polarity,scl_valence
9638,typhus,theme_or_plot,negative,-1.000000
9636,slavery,theme_or_plot,negative,-0.896000
9632,civil war,theme_or_plot,negative,-0.534000
9639,widow,theme_or_plot,negative,-0.500000
9642,casualty of war,plot_event,negative,-0.495333
9644,american civil war,plot_event,negative,-0.429500
9640,marriage crisis,theme_or_plot,negative,-0.172500
9649,antebellum south,theme_or_plot,positive,0.005500
9633,loss of loved one,plot_event,positive,0.066500
9637,plantation,theme_or_plot,positive,0.146000


## Dashboard Layout & App

In [92]:
# ── Layout ────────────────────────────────────────────────────────

MIN_YEAR, MAX_YEAR = int(movies_df['year'].min()), int(movies_df['year'].max())

def create_layout():
    """Builds the Dash layout: header, sticky filter card, and four chart containers."""
    return dbc.Container([

        html.H2('In the Mood for Love: Movie Sentiment & Box Office Revenue', style={'textAlign': 'center', 'paddingTop': '20px', 'marginBottom': '5px'}),
        html.P(
            'Which films do better at the box office? Using sentiment classifications derived from lexicon analysis of TMDB movie keywords, this dashboard maps narrative tone against gross revenue to explore whether sentiment has any bearing on commercial success.',
            style={'textAlign': 'center', 'color': COLORS['text'], 'maxWidth': '720px', 'margin': '0 auto 20px', 'fontSize': '0.95rem'},
        ),

        # Sticky filter card with year range slider and reset button
        dbc.Card([
            dbc.CardBody(
                dbc.Row([
                    dbc.Col([
                        html.Label('Year Range', className='fw-bold small mb-1'),
                        dcc.RangeSlider(
                            id='year-slider',
                            min=MIN_YEAR, max=MAX_YEAR, step=1,
                            value=[1960, MAX_YEAR],
                            marks={y: str(y) for y in range(MIN_YEAR, MAX_YEAR + 1, 20)},
                        ),
                    ], width=10),
                    dbc.Col([
                        dbc.Button('Reset Filters', id='reset-btn', color='secondary',
                                   size='sm', className='mt-3'),
                    ], width=2, className='d-flex align-items-center justify-content-end'),
                ], align='center')
            ),
        ], style={
            'marginBottom': '20px',
            'position': 'sticky',
            'top': '0',
            'zIndex': '1000',
            'boxShadow': '0 2px 8px rgba(0,0,0,0.12)',
        }),

        dcc.Graph(id='area-chart'),
        dcc.Graph(id='treemap-chart'),

        dbc.Row([
            dbc.Col(dcc.Graph(id='genre-chart'), width=6),
            dbc.Col(dcc.Graph(id='keyword-chart'), width=6),
        ])

    ], fluid=False, style={'backgroundColor': COLORS['bg'], 'padding': '20px', '--rc-slider-color': '#333'})

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────

def register_callbacks(app):
    """Cross-filter callback: genre click filters all views, treemap click
    filters keywords, keyword click filters treemap. Reset clears everything.
    """
    @app.callback(
        Output('area-chart', 'figure'),
        Output('treemap-chart', 'figure'),
        Output('genre-chart', 'figure'),
        Output('keyword-chart', 'figure'),
        Input('year-slider', 'value'),
        Input('genre-chart', 'clickData'),
        Input('treemap-chart', 'clickData'),
        Input('keyword-chart', 'clickData'),
        Input('reset-btn', 'n_clicks'),
    )
    def update_charts(year_range, genre_click, treemap_click, kw_click, n_clicks):
        trigger = ctx.triggered_id
        year_filtered = movies_df[movies_df['year'].between(year_range[0], year_range[1])]
        filtered = year_filtered

        selected_genre = None
        selected_movie_id = None
        selected_keyword = None

        # Handle genre click - update treemap, keywords
        if trigger == 'genre-chart' and genre_click:
            selected_genre = genre_click['points'][0]['customdata']
            filtered = year_filtered[year_filtered['genres'].str.contains(selected_genre, na=False)]

        # Handle treemap click - show keywords for the movie
        if trigger == 'treemap-chart' and treemap_click:
            label = treemap_click['points'][0]['label']
            match = filtered[filtered['title'] == label]
            if not match.empty:
                selected_movie_id = match.iloc[0]['tmdb_movie_id']

        # Handle Keyword Click - filter treemap 
        if trigger == 'keyword-chart' and kw_click:
            selected_keyword = kw_click['points'][0]['y']

        return (
            make_area_chart(filtered),
            make_treemap(filtered, year_range, selected_keyword=selected_keyword),
            make_genre_chart(year_filtered, year_range, selected_genre=selected_genre),
            make_keyword_chart(filtered, year_range, selected_movie_id=selected_movie_id),
        )

In [94]:
# ── Run ───────────────────────────────────────────────────────────

EXT = [
    dbc.themes.LUX,
    "https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css",
]

app = Dash(__name__, external_stylesheets=EXT, suppress_callback_exceptions=True)
app.title = "Movie Keyword Sentiment and Box Office Revenue"
app.layout = create_layout()
register_callbacks(app)

app.run(jupyter_mode='inline', debug=False)

## Exploratory Lookups

In [95]:
movie = movies_df[movies_df['title'] == 'Memento'].iloc[0]
print(f"Title: {movie['title']}")
print(f"Sentiment Mean: {movie['sentiment_mean']:.4f}")
print(f"Sentiment Category: {movie['sentiment_category']}")
print(f"Revenue: ${movie['revenue']:,.0f}")
print()

movie_kw = keywords_df[keywords_df['tmdb_movie_id'] == movie['tmdb_movie_id']]
print(f"Keywords ({len(movie_kw)}):")
movie_kw[['name', 'keyword_type', 'scl_polarity', 'scl_valence']].sort_values('scl_valence')

Title: Memento
Sentiment Mean: -0.1723
Sentiment Category: negative
Revenue: $39,723,096

Keywords (19):


,name,keyword_type,scl_polarity,scl_valence
725,murder,theme_or_plot,negative,-0.986
724,revenge,theme_or_plot,negative,-0.766
720,drug dealer,theme_or_plot,negative,-0.700
728,memory loss,theme_or_plot,negative,-0.646
727,confusion,theme_or_plot,negative,-0.524
722,manipulation,theme_or_plot,negative,-0.515
726,flashback,theme_or_plot,negative,-0.470
719,amnesia,theme_or_plot,negative,-0.408
715,insulin,theme_or_plot,negative,-0.142
716,tattoo,theme_or_plot,negative,-0.020


In [96]:
movie = movies_df[movies_df['title'] == 'Forrest Gump'].iloc[0]
print(f"Title: {movie['title']}")
print(f"Sentiment Mean: {movie['sentiment_mean']:.4f}")
print(f"Sentiment Category: {movie['sentiment_category']}")
print(f"Revenue: ${movie['revenue']:,.0f}")
print()

movie_kw = keywords_df[keywords_df['tmdb_movie_id'] == movie['tmdb_movie_id']]
print(f"Keywords ({len(movie_kw)}):")
movie_kw[['name', 'keyword_type', 'scl_polarity', 'scl_valence']].sort_values('scl_valence')

Title: Forrest Gump
Sentiment Mean: 0.0950
Sentiment Category: positive
Revenue: $677,387,716

Keywords (26):


,name,keyword_type,scl_polarity,scl_valence
78,drug addiction,theme_or_plot,negative,-0.922000
80,vietnam war,theme_or_plot,negative,-0.666000
66,post-traumatic stress disorder (ptsd),plot_event,negative,-0.598500
77,autism,theme_or_plot,negative,-0.530000
87,anti war protest,plot_event,negative,-0.471333
86,military,theme_or_plot,negative,-0.208000
70,mentally disabled,theme_or_plot,negative,-0.192000
65,vietnam veteran,theme_or_plot,negative,-0.162000
82,bus stop,theme_or_plot,negative,-0.134000
69,single parent,theme_or_plot,neutral,0.000000
